# Notebook 03 — IRT Model Fitting

**Goal**: Fit 1PL, 2PL, 3PL IRT models per sample, select best model, extract item parameters,
compute item fit statistics, and plot ICCs. Real-author items only.

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import girth

warnings.filterwarnings('ignore')

_MARKER = "data/processed/irt_pipeline/sample1_pretest.csv"
PROJECT_ROOT = os.path.abspath(os.getcwd())
while PROJECT_ROOT:
    if os.path.isfile(os.path.join(PROJECT_ROOT, _MARKER)):
        break
    _parent = os.path.dirname(PROJECT_ROOT)
    if _parent == PROJECT_ROOT:
        PROJECT_ROOT = os.path.abspath(os.getcwd())
        break
    PROJECT_ROOT = _parent

DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed/irt_pipeline")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/03_irt_fitting")
os.makedirs(RESULTS_DIR, exist_ok=True)

s1_auth = pd.read_csv(os.path.join(DATA_DIR, "sample1_authors_only.csv"))
s2_auth = pd.read_csv(os.path.join(DATA_DIR, "sample2_authors_only.csv"))
meta = pd.read_csv(os.path.join(DATA_DIR, "item_metadata.csv"))

s1_auth = s1_auth.apply(pd.to_numeric, errors='coerce')
s2_auth = s2_auth.apply(pd.to_numeric, errors='coerce')
author_cols = s1_auth.columns.tolist()

# Prepare clean integer arrays (girth requirement)
X1 = s1_auth.dropna().values.astype(int)
X2 = s2_auth.dropna().values.astype(int)
print(f"Sample 1: {X1.shape}, Sample 2: {X2.shape}")

Sample 1: (449, 101), Sample 2: (798, 101)


---
## 3a. IRT Model Fitting (1PL, 2PL, 3PL per sample)

In [2]:
def fit_all_models(X, name):
    """Fit 1PL, 2PL, and 3PL and return results."""
    results = {}
    n_persons, n_items = X.shape
    
    # 1PL (Rasch)
    print(f"\n--- {name}: Fitting 1PL ---")
    r1 = girth.rasch_mml(X.T)
    results['1PL'] = {
        'b': r1['Difficulty'],
        'a': np.full(n_items, r1.get('Discrimination', 1.0) if np.isscalar(r1.get('Discrimination', 1.0)) else r1['Discrimination'].mean()),
        'n_params': n_items + 1
    }
    print(f"  b: M = {r1['Difficulty'].mean():.3f}, SD = {r1['Difficulty'].std():.3f}")
    
    # 2PL
    print(f"\n--- {name}: Fitting 2PL ---")
    r2 = girth.twopl_mml(X.T)
    results['2PL'] = {
        'a': r2['Discrimination'],
        'b': r2['Difficulty'],
        'n_params': 2 * n_items
    }
    print(f"  a: M = {r2['Discrimination'].mean():.3f}, SD = {r2['Discrimination'].std():.3f}")
    print(f"  b: M = {r2['Difficulty'].mean():.3f}, SD = {r2['Difficulty'].std():.3f}")
    
    # 3PL
    print(f"\n--- {name}: Fitting 3PL ---")
    try:
        r3 = girth.threepl_mml(X.T)
        results['3PL'] = {
            'a': r3['Discrimination'],
            'b': r3['Difficulty'],
            'c': r3.get('Guessing', np.zeros(n_items)),
            'n_params': 3 * n_items
        }
        print(f"  a: M = {r3['Discrimination'].mean():.3f}")
        print(f"  b: M = {r3['Difficulty'].mean():.3f}")
        c_vals = results['3PL']['c']
        if hasattr(c_vals, 'mean'):
            print(f"  c: M = {c_vals.mean():.3f}")
    except Exception as e:
        print(f"  3PL did not converge: {e}")
        results['3PL'] = None
    
    return results

models1 = fit_all_models(X1, 'Sample 1')
models2 = fit_all_models(X2, 'Sample 2')


--- Sample 1: Fitting 1PL ---
  b: M = -0.244, SD = 1.834

--- Sample 1: Fitting 2PL ---


  a: M = 1.681, SD = 0.718
  b: M = 0.064, SD = 1.748

--- Sample 1: Fitting 3PL ---


  3PL did not converge: setting an array element with a sequence.

--- Sample 2: Fitting 1PL ---
  b: M = 0.692, SD = 2.377

--- Sample 2: Fitting 2PL ---


  a: M = 1.544, SD = 0.812
  b: M = 1.231, SD = 2.816

--- Sample 2: Fitting 3PL ---


  3PL did not converge: setting an array element with a sequence.


---
## 3b. Model Selection (AIC/BIC)

In [3]:
def compute_log_likelihood(X, a, b, c=None):
    """Compute marginal log-likelihood for IRT model."""
    n_persons, n_items = X.shape
    theta = girth.ability_eap(X.T, b, a)
    
    ll = 0.0
    for i in range(n_persons):
        for j in range(n_items):
            p = 1.0 / (1.0 + np.exp(-a[j] * (theta[i] - b[j])))
            if c is not None:
                p = c[j] + (1.0 - c[j]) * p
            p = np.clip(p, 1e-10, 1 - 1e-10)
            ll += X[i, j] * np.log(p) + (1 - X[i, j]) * np.log(1 - p)
    return ll

def model_comparison(X, models, name):
    """Compare models using AIC/BIC."""
    n = X.shape[0]
    
    print(f"\n{'='*60}")
    print(f"Model Comparison: {name}")
    print(f"{'='*60}")
    print(f"{'Model':<8} {'n_params':>8} {'LL':>12} {'AIC':>12} {'BIC':>12}")
    print("-" * 54)
    
    results = {}
    for mname in ['1PL', '2PL', '3PL']:
        if models[mname] is None:
            print(f"{mname:<8} {'N/A':>8} {'N/A':>12} {'N/A':>12} {'N/A':>12}")
            continue
        
        m = models[mname]
        c = m.get('c', None)
        ll = compute_log_likelihood(X, m['a'], m['b'], c)
        k = m['n_params']
        aic = -2 * ll + 2 * k
        bic = -2 * ll + np.log(n) * k
        
        results[mname] = {'LL': ll, 'AIC': aic, 'BIC': bic, 'k': k}
        print(f"{mname:<8} {k:>8d} {ll:>12.1f} {aic:>12.1f} {bic:>12.1f}")
    
    # LR test: 1PL vs 2PL
    if '1PL' in results and '2PL' in results:
        lr = -2 * (results['1PL']['LL'] - results['2PL']['LL'])
        df = results['2PL']['k'] - results['1PL']['k']
        p_lr = 1 - stats.chi2.cdf(lr, df)
        print(f"\nLR test 1PL vs 2PL: chi2({df}) = {lr:.1f}, p = {p_lr:.2e}")
        if p_lr < 0.05:
            print("  2PL significantly better than 1PL")
    
    # LR test: 2PL vs 3PL
    if '2PL' in results and '3PL' in results:
        lr = -2 * (results['2PL']['LL'] - results['3PL']['LL'])
        df = results['3PL']['k'] - results['2PL']['k']
        p_lr = 1 - stats.chi2.cdf(lr, df)
        print(f"LR test 2PL vs 3PL: chi2({df}) = {lr:.1f}, p = {p_lr:.2e}")
    
    # Best model by BIC
    best = min(results.keys(), key=lambda k: results[k]['BIC'])
    print(f"\nBest model by BIC: {best}")
    
    return results, best

comp1, best1 = model_comparison(X1, models1, 'Sample 1')
comp2, best2 = model_comparison(X2, models2, 'Sample 2')


Model Comparison: Sample 1
Model    n_params           LL          AIC          BIC
------------------------------------------------------


1PL           102     -18145.2      36494.5      36913.4


2PL           202     -17392.7      35189.5      36019.1
3PL           N/A          N/A          N/A          N/A

LR test 1PL vs 2PL: chi2(100) = 1505.0, p = 0.00e+00
  2PL significantly better than 1PL

Best model by BIC: 2PL

Model Comparison: Sample 2
Model    n_params           LL          AIC          BIC
------------------------------------------------------


1PL           102     -27695.1      55594.1      56071.7


2PL           202     -26348.8      53101.7      54047.5
3PL           N/A          N/A          N/A          N/A

LR test 1PL vs 2PL: chi2(100) = 2692.4, p = 0.00e+00
  2PL significantly better than 1PL

Best model by BIC: 2PL


---
## 3c. Item Parameters (from 2PL)

In [4]:
def item_parameter_table(models, author_cols, meta, sample_name):
    """Build item parameter table from 2PL."""
    m = models['2PL']
    a, b = m['a'], m['b']
    
    # Get genre codes from metadata
    code_map = dict(zip(meta['item_label'], meta['item_code']))
    
    tbl = pd.DataFrame({
        'item': author_cols,
        'genre': [code_map.get(c, '?') for c in author_cols],
        'a': a,
        'b': b,
    })
    tbl['a_flag'] = tbl['a'] < 0.50
    tbl['b_extreme'] = tbl['b'].abs() > 3.0
    
    print(f"\n{'='*60}")
    print(f"Item Parameters (2PL): {sample_name}")
    print(f"{'='*60}")
    print(f"  Discrimination (a): M = {a.mean():.3f}, SD = {a.std():.3f}, "
          f"Range = [{a.min():.3f}, {a.max():.3f}]")
    print(f"  Difficulty (b): M = {b.mean():.3f}, SD = {b.std():.3f}, "
          f"Range = [{b.min():.3f}, {b.max():.3f}]")
    print(f"  Items with a < 0.50 (poor): {tbl['a_flag'].sum()}")
    print(f"  Items with |b| > 3.0 (extreme): {tbl['b_extreme'].sum()}")
    
    return tbl

params1 = item_parameter_table(models1, author_cols, meta, 'Sample 1')
params2 = item_parameter_table(models2, author_cols, meta, 'Sample 2')

params1.to_csv(os.path.join(RESULTS_DIR, 'item_params_2pl_sample1.csv'), index=False)
params2.to_csv(os.path.join(RESULTS_DIR, 'item_params_2pl_sample2.csv'), index=False)

# Parameter distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (p, name) in enumerate([(params1, 'Sample 1'), (params2, 'Sample 2')]):
    axes[i, 0].hist(p['a'], bins=25, edgecolor='black', alpha=0.7)
    axes[i, 0].set_title(f'{name}: Discrimination (a)')
    axes[i, 0].axvline(0.5, color='red', linestyle='--', label='a=0.50 threshold')
    axes[i, 0].legend()
    
    axes[i, 1].hist(p['b'], bins=25, edgecolor='black', alpha=0.7)
    axes[i, 1].set_title(f'{name}: Difficulty (b)')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'item_parameter_distributions.png'), dpi=150)
plt.show()


Item Parameters (2PL): Sample 1
  Discrimination (a): M = 1.681, SD = 0.718, Range = [0.372, 3.783]
  Difficulty (b): M = 0.064, SD = 1.748, Range = [-3.797, 5.997]
  Items with a < 0.50 (poor): 5
  Items with |b| > 3.0 (extreme): 10

Item Parameters (2PL): Sample 2
  Discrimination (a): M = 1.544, SD = 0.812, Range = [0.259, 3.981]
  Difficulty (b): M = 1.231, SD = 2.816, Range = [-5.967, 5.995]
  Items with a < 0.50 (poor): 6
  Items with |b| > 3.0 (extreme): 27


---
## 3d. Item Fit Statistics

In [5]:
def compute_item_fit(X, a, b, name, n_groups=10):
    """Compute infit/outfit MNSQ and S-X2 item fit statistics."""
    n_persons, n_items = X.shape
    theta = girth.ability_eap(X.T, b, a)
    
    # Expected probabilities
    P = np.zeros_like(X, dtype=float)
    for j in range(n_items):
        P[:, j] = 1.0 / (1.0 + np.exp(-a[j] * (theta - b[j])))
    P = np.clip(P, 1e-10, 1 - 1e-10)
    
    # Residuals
    resid = X - P
    var_ij = P * (1 - P)
    std_resid = resid / np.sqrt(var_ij)
    
    # Outfit MNSQ: mean of squared standardized residuals per item
    outfit = np.mean(std_resid**2, axis=0)
    
    # Infit MNSQ: variance-weighted mean square
    infit = np.sum(resid**2, axis=0) / np.sum(var_ij, axis=0)
    
    # S-X2 statistic (Orlando & Thissen, 2000)
    # Group persons by theta, compare observed vs expected proportions
    theta_groups = pd.qcut(theta, n_groups, labels=False, duplicates='drop')
    actual_groups = len(np.unique(theta_groups))
    
    s_x2 = np.zeros(n_items)
    s_x2_df = np.zeros(n_items, dtype=int)
    
    for j in range(n_items):
        chi2_j = 0.0
        df_j = 0
        for g in np.unique(theta_groups):
            mask = theta_groups == g
            n_g = mask.sum()
            if n_g < 5:
                continue
            obs = X[mask, j].mean()
            exp = P[mask, j].mean()
            if exp > 0 and exp < 1:
                chi2_j += n_g * (obs - exp)**2 / (exp * (1 - exp))
                df_j += 1
        s_x2[j] = chi2_j
        s_x2_df[j] = max(df_j - 1, 1)  # df = groups - 1 (minus estimated params handled externally)
    
    s_x2_p = 1 - stats.chi2.cdf(s_x2, s_x2_df)
    bonf = 0.05 / n_items
    
    print(f"\n{'='*60}")
    print(f"Item Fit Statistics: {name}")
    print(f"{'='*60}")
    print(f"  Infit MNSQ:  M = {infit.mean():.3f}, SD = {infit.std():.3f}, "
          f"Range = [{infit.min():.3f}, {infit.max():.3f}]")
    print(f"  Outfit MNSQ: M = {outfit.mean():.3f}, SD = {outfit.std():.3f}, "
          f"Range = [{outfit.min():.3f}, {outfit.max():.3f}]")
    print(f"  Items with infit > 1.5: {(infit > 1.5).sum()}")
    print(f"  Items with outfit > 2.0: {(outfit > 2.0).sum()}")
    print(f"\n  S-X2 statistics:")
    print(f"  Items significant (p < {bonf:.2e}, Bonferroni): {(s_x2_p < bonf).sum()}")
    
    return pd.DataFrame({
        'item': author_cols,
        'infit': infit,
        'outfit': outfit,
        'S_X2': s_x2,
        'S_X2_df': s_x2_df,
        'S_X2_p': s_x2_p,
        'S_X2_sig_bonf': s_x2_p < bonf
    })

fit1 = compute_item_fit(X1, models1['2PL']['a'], models1['2PL']['b'], 'Sample 1')
fit2 = compute_item_fit(X2, models2['2PL']['a'], models2['2PL']['b'], 'Sample 2')

fit1.to_csv(os.path.join(RESULTS_DIR, 'item_fit_sample1.csv'), index=False)
fit2.to_csv(os.path.join(RESULTS_DIR, 'item_fit_sample2.csv'), index=False)


Item Fit Statistics: Sample 1
  Infit MNSQ:  M = 0.995, SD = 0.060, Range = [0.919, 1.304]
  Outfit MNSQ: M = 1.066, SD = 0.866, Range = [0.355, 8.911]
  Items with infit > 1.5: 0
  Items with outfit > 2.0: 3

  S-X2 statistics:
  Items significant (p < 4.95e-04, Bonferroni): 0

Item Fit Statistics: Sample 2
  Infit MNSQ:  M = 0.991, SD = 0.028, Range = [0.904, 1.101]
  Outfit MNSQ: M = 1.108, SD = 0.847, Range = [0.454, 8.873]
  Items with infit > 1.5: 0
  Items with outfit > 2.0: 2

  S-X2 statistics:
  Items significant (p < 4.95e-04, Bonferroni): 2


---
## 3e. Item Characteristic Curves (ICCs)

In [6]:
def plot_iccs(X, a, b, author_cols, name, n_show=20):
    """Plot ICCs with observed proportions."""
    theta_est = girth.ability_eap(X.T, b, a)
    theta_range = np.linspace(-4, 4, 200)
    
    # Select items spread across difficulty range
    sorted_idx = np.argsort(b)
    step = max(1, len(sorted_idx) // n_show)
    selected = sorted_idx[::step][:n_show]
    
    n_rows = (len(selected) + 3) // 4
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 3 * n_rows))
    axes = axes.flatten()
    
    for idx, item_j in enumerate(selected):
        ax = axes[idx]
        
        # Theoretical ICC
        p_curve = 1.0 / (1.0 + np.exp(-a[item_j] * (theta_range - b[item_j])))
        ax.plot(theta_range, p_curve, 'b-', linewidth=2)
        
        # Observed proportions in theta bins
        n_bins = 10
        bin_edges = np.linspace(theta_est.min(), theta_est.max(), n_bins + 1)
        for bi in range(n_bins):
            mask = (theta_est >= bin_edges[bi]) & (theta_est < bin_edges[bi + 1])
            if mask.sum() > 5:
                obs_p = X[mask, item_j].mean()
                mid = (bin_edges[bi] + bin_edges[bi + 1]) / 2
                ax.plot(mid, obs_p, 'ro', markersize=4)
        
        item_name = author_cols[item_j] if item_j < len(author_cols) else f"Item {item_j}"
        ax.set_title(f"{item_name[:20]}\na={a[item_j]:.2f}, b={b[item_j]:.2f}", fontsize=8)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('θ', fontsize=7)
    
    for idx in range(len(selected), len(axes)):
        axes[idx].set_visible(False)
    
    fig.suptitle(f'{name}: Item Characteristic Curves (2PL)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'iccs_{name.lower().replace(" ", "_")}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

plot_iccs(X1, models1['2PL']['a'], models1['2PL']['b'], author_cols, 'Sample 1')
plot_iccs(X2, models2['2PL']['a'], models2['2PL']['b'], author_cols, 'Sample 2')

---
## Summary

In [7]:
# Save 2PL parameters for downstream notebooks
np.savez(os.path.join(RESULTS_DIR, 'irt_2pl_params_sample1.npz'),
         a=models1['2PL']['a'], b=models1['2PL']['b'])
np.savez(os.path.join(RESULTS_DIR, 'irt_2pl_params_sample2.npz'),
         a=models2['2PL']['a'], b=models2['2PL']['b'])

print("="*60)
print("IRT MODEL FITTING COMPLETE")
print("="*60)
print(f"\nBest model by BIC: Sample 1 = {best1}, Sample 2 = {best2}")
print(f"\n2PL parameters saved to: {RESULTS_DIR}")
print(f"\nFiles:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {f}")

IRT MODEL FITTING COMPLETE

Best model by BIC: Sample 1 = 2PL, Sample 2 = 2PL

2PL parameters saved to: /home/polina/Documents/Cursor_Projects/Russian Author Recognition Test Cursor/data/processed/results/03_irt_fitting

Files:
  iccs_sample_1.png
  iccs_sample_2.png
  irt_2pl_params_sample1.npz
  irt_2pl_params_sample2.npz
  item_fit_sample1.csv
  item_fit_sample2.csv
  item_parameter_distributions.png
  item_params_2pl_sample1.csv
  item_params_2pl_sample2.csv
